## 연습 1: Northbank Financial 데이터 모델 생성

이 연습에서는 Northbank Financial의 분석 플랫폼을 위한 기초 Delta 테이블을 설계합니다. 이 모델에는 SCD Type 2 기록 추적을 위해 설계된 **고객 차원** 및 액상 클러스터링과 변경 데이터 피드로 최적화된 **트랜잭션 팩트 테이블**이 포함됩니다.

이 연습을 완료한 후, 랩 설정 페이지로 돌아가서 **Catalog Explorer**에서 생성한 테이블을 살펴봅니다.

### 작업 1.1 — 카탈로그 및 스키마 생성

Unity Catalog **카탈로그**를 `banking_lab`으로 명명하고 적절한 주석을 포함합니다. 그 안에 `silver`라는 **스키마**를 생성합니다.

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS banking_lab
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Northbank Financial lakehouse platform for customer analytics and regulatory reporting'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS banking_lab.silver
  COMMENT 'Cleansed and enriched banking data — customer dimensions and transaction facts';

### 작업 1.2 — `dim_customer` SCD Type 2 테이블 생성

managed 델타 테이블 `banking_lab.silver.dim_customer`를 생성합니다. 이 테이블은 SCD Type 2 패턴을 사용하여 **고객 차원 기록**을 추적합니다.

테이블에는 다음이 포함되어야 합니다:

| 열 | 타입 | 참고 |
|---|---|---|
| `customer_sk` | `BIGINT GENERATED ALWAYS AS IDENTITY` | 서로게이트 키 — 자동 증가, 버전당 고유 |
| `customer_id` | `STRING NOT NULL` | 원본 시스템의 비즈니스/자연 키 |
| `full_name` | `STRING` | |
| `email` | `STRING` | |
| `city` | `STRING` | 주거지 도시 |
| `segment` | `STRING` | 고객 세그먼트: Retail, Premium, Business |
| `account_type` | `STRING` | 계정 타입: Current, Savings, Business |
| `valid_from` | `TIMESTAMP NOT NULL` | 이 버전이 유효해진 시점 |
| `valid_to` | `TIMESTAMP NOT NULL` | 이 버전이 만료된 시점 (활성 기록의 경우 9999-12-31) |
| `is_current` | `BOOLEAN` | 활성 버전인 경우 True |

`customer_id`에 **액상 클러스터링**을 적용하고 **변경 데이터 피드**를 활성화합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I create a Delta table in Databricks with GENERATED ALWAYS AS IDENTITY, CLUSTER BY, and Change Data Feed enabled in TBLPROPERTIES?"*

**힌트:**
- `CLUSTER BY (customer_id)`를 사용하여 조회 성능을 최적화합니다
- `TBLPROPERTIES (delta.enableChangeDataFeed = true)`를 추가하여 감사 추적을 지원합니다
- `USING DELTA`를 사용합니다 (Azure Databricks의 모든 테이블에 권장됨)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS banking_lab.silver.dim_customer (
    customer_sk  BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id  STRING NOT NULL,
    full_name    STRING,
    email        STRING,
    city         STRING,
    segment      STRING,
    account_type STRING,
    valid_from   TIMESTAMP NOT NULL,
    valid_to     TIMESTAMP NOT NULL,
    is_current   BOOLEAN
)
USING DELTA
CLUSTER BY (customer_id)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

### 작업 1.3 — `fact_transactions` 테이블 생성

개별 은행 결제를 저장하기 위해 managed 델타 테이블 `banking_lab.silver.fact_transactions`를 생성합니다.

| 열 | 타입 | 참고 |
|---|---|---|
| `transaction_id` | `STRING NOT NULL` | 고유 거래 참조 |
| `account_id` | `STRING` | 은행 계정 식별자 |
| `customer_id` | `STRING` | `dim_customer`로 링크 |
| `transaction_type` | `STRING` | Credit, Debit, 또는 Transfer |
| `amount` | `DECIMAL(15,2)` | 거래 금액 |
| `currency` | `STRING DEFAULT 'GBP'` | 통화 코드 |
| `transaction_date` | `DATE` | 거래 날짜 |
| `description` | `STRING` | 거래 설명 |

예상되는 쿼리 패턴(고객 및/또는 날짜 범위로 필터링)을 지원하기 위해 `customer_id, transaction_date`에 **액상 클러스터링**을 적용합니다. FCA 규정 준수 감사를 위해 **변경 데이터 피드**를 활성화합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I create a Delta Lake table in Databricks SQL with liquid clustering and Change Data Feed enabled?"*

**힌트:**
- `CLUSTER BY (customer_id, transaction_date)`를 사용합니다 — 최대 4개의 클러스터링 키를 지정할 수 있습니다
- `TBLPROPERTIES (delta.enableChangeDataFeed = true)`를 추가합니다
- currency 열에 `STRING DEFAULT 'GBP'`를 사용합니다

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS banking_lab.silver.fact_transactions (
    transaction_id   STRING NOT NULL,
    account_id       STRING,
    customer_id      STRING,
    transaction_type STRING,
    amount           DECIMAL(15,2),
    currency         STRING DEFAULT 'GBP',
    transaction_date DATE,
    description      STRING
)
USING DELTA
CLUSTER BY (customer_id, transaction_date)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true,
    delta.feature.allowColumnDefaults = 'supported'
);

## 연습 2: 고객 차원을 위한 SCD Type 2 구현

Northbank의 규제 보고는 현재 상태가 아니라 **각 거래 시점의 고객 프로필**을 반영해야 합니다. 이 연습에서는 초기 고객 기록을 로드하고, 실제 특성 변경을 적용하며, SCD Type 2를 위한 **2단계 MERGE + INSERT 패턴**을 구현합니다. 그런 다음 전체 고객 기록을 조회하고 특정 시점의 조회를 수행합니다.

### 작업 2.1 — 초기 고객 기록 로드

아래 셀에서는 8명의 Northbank 고객에 대한 **초기 데이터 세트**를 PySpark DataFrame으로 정의합니다. 작업은 이 DataFrame을 **append 모드에서 `banking_lab.silver.dim_customer`에 작성**하는 것입니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I write a PySpark DataFrame to an existing Delta table in Databricks using saveAsTable in append mode?"*

**힌트:**
- `df.write.mode("append").saveAsTable("banking_lab.silver.dim_customer")`를 사용합니다
- 데이터와 DataFrame 설정은 이미 제공되었습니다 — DataFrame이 빌드된 후에 write 문을 추가합니다

In [ ]:
from pyspark.sql.functions import lit, to_timestamp

initial_customers = [
    ("C-1001", "Emma Hartley",      "emma.hartley@northbank.com",     "London",     "Retail",   "Savings"),
    ("C-1002", "James Weston",      "james.weston@northbank.com",     "Manchester", "Retail",   "Current"),
    ("C-1003", "Sophia Chen",       "sophia.chen@northbank.com",      "Edinburgh",  "Premium",  "Savings"),
    ("C-1004", "Oliver Banks",      "oliver.banks@northbank.com",     "Birmingham", "Retail",   "Current"),
    ("C-1005", "Aisha Patel",       "aisha.patel@northbank.com",      "London",     "Business", "Business"),
    ("C-1006", "Liam Murray",       "liam.murray@northbank.com",      "Leeds",      "Retail",   "Savings"),
    ("C-1007", "Charlotte Wright",  "charlotte.wright@northbank.com", "Bristol",    "Premium",  "Current"),
    ("C-1008", "Noah Thompson",     "noah.thompson@northbank.com",    "Glasgow",    "Retail",   "Savings"),
]

df_initial = spark.createDataFrame(
    initial_customers,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)

df_initial = (
    df_initial
    .withColumn("valid_from",  to_timestamp(lit("2020-01-01 00:00:00")))
    .withColumn("valid_to",    to_timestamp(lit("9999-12-31 00:00:00")))
    .withColumn("is_current",  lit(True))
)

df_initial.write.mode("append").saveAsTable("banking_lab.silver.dim_customer")
print(f"Loaded {df_initial.count()} initial customer records.")

In [ ]:
%sql
-- Verify: you should see 8 customer records
SELECT customer_sk, customer_id, full_name, city, segment, is_current
FROM banking_lab.silver.dim_customer
ORDER BY customer_sk;

### 작업 2.2 — 고객 변경사항이 포함된 스테이징 데이터 로드

운영 팀이 다음 변경 알림을 받았습니다:

| 고객 | 변경 |
|---|---|
| **C-1001 Emma Hartley** | **London → Oxford**로 이주, **Premium** 세그먼트로 업그레이드 |
| **C-1003 Sophia Chen** | **Edinburgh → London**으로 이주 |
| **C-1006 Liam Murray** | 이메일 주소만 업데이트 (도시 및 세그먼트는 변경 없음) |
| **C-1009 Priya Singh** | **새로운 고객** Cardiff에서 가입, Retail 세그먼트 |

아래 셀을 실행하여 스테이징 DataFrame을 생성하고 임시 뷰로 등록합니다. 다음 작업에서 이를 사용합니다.

In [ ]:
staging_data = [
    ("C-1001", "Emma Hartley",    "emma.hartley@northbank.com",     "Oxford",   "Premium",  "Savings"),
    ("C-1003", "Sophia Chen",     "sophia.chen@northbank.com",      "London",   "Premium",  "Savings"),
    ("C-1006", "Liam Murray",     "liam.new@northbank.com",         "Leeds",    "Retail",   "Savings"),
    ("C-1009", "Priya Singh",     "priya.singh@northbank.com",      "Cardiff",  "Retail",   "Current"),
]

df_staging = spark.createDataFrame(
    staging_data,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)
df_staging.createOrReplaceTempView("staging_customers")
print("Staging data loaded. Temp view 'staging_customers' is available.")
display(df_staging)

### 작업 2.3 — SCD Type 2 적용: 단계 1 — 변경된 기록 종료

`staging_customers`를 `dim_customer`의 현재 버전과 비교하는 `MERGE INTO` 문을 작성합니다. **추적된 특성**(도시, 세그먼트, 이메일, 계정 타입) 중 하나라도 변경된 기록의 경우 다음을 설정하여 **현재 버전을 종료**합니다:
- `valid_to = current_timestamp()`
- `is_current = false`

> 🤖 **Genie Code 팁:** 요청하세요:
> *"Write a Databricks SQL MERGE statement that closes SCD Type 2 dimension records by setting is_current=false and valid_to=current_timestamp() when any tracked attribute changes."*

**힌트:**
- `target.customer_id = source.customer_id AND target.is_current = true`에서 일치시킵니다
- `WHEN MATCHED AND (...)`  조건은 **추적된 특성 중 하나라도** 다른지 확인해야 합니다
- `valid_to`와 `is_current`만 업데이트합니다 — 기록 데이터 열을 덮어쓰지 마십시오

In [ ]:
%sql
-- Step 1: Close the current version of changed customer records
MERGE INTO banking_lab.silver.dim_customer AS target
USING staging_customers AS source
ON target.customer_id = source.customer_id
   AND target.is_current = true
WHEN MATCHED AND (
    target.city         <> source.city         OR
    target.segment      <> source.segment       OR
    target.email        <> source.email         OR
    target.account_type <> source.account_type
) THEN UPDATE SET
    target.valid_to    = current_timestamp(),
    target.is_current  = false;

### 작업 2.4 — SCD Type 2 적용: 단계 2 — 새로운 버전 및 새로운 고객 삽입

`INSERT INTO` 문을 작성하여:
1. 방금 종료된 이전 버전이 있는 고객(C-1001, C-1003, C-1006)에 대해 **새로운 현재 행** 추가
2. 기존 기록이 없는 **새로운 고객** C-1009 추가

새로운 행은 `valid_from = current_timestamp()`, `valid_to = TIMESTAMP '9999-12-31'`, `is_current = true`를 가져야 합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"Write a Databricks SQL INSERT statement to add new current SCD Type 2 rows from a staging table, only for customers where no is_current=true record exists."*

**힌트:** `WHERE NOT EXISTS (SELECT 1 FROM banking_lab.silver.dim_customer d WHERE d.customer_id = s.customer_id AND d.is_current = true)`를 사용하여 새 행을 삽입해야 하는 스테이징 기록을 식별합니다.

In [ ]:
%sql
-- Step 2: Insert new current versions for changed customers and the new customer
INSERT INTO banking_lab.silver.dim_customer
    (customer_id, full_name, email, city, segment, account_type, valid_from, valid_to, is_current)
SELECT
    s.customer_id,
    s.full_name,
    s.email,
    s.city,
    s.segment,
    s.account_type,
    current_timestamp()              AS valid_from,
    TIMESTAMP '9999-12-31 00:00:00'  AS valid_to,
    true                             AS is_current
FROM staging_customers s
WHERE NOT EXISTS (
    SELECT 1
    FROM banking_lab.silver.dim_customer d
    WHERE d.customer_id = s.customer_id
      AND d.is_current = true
);

### 작업 2.5 — 고객의 전체 버전 기록 조회

**모든 버전**의 고객 `C-1001` (Emma Hartley)을 `valid_from`으로 정렬하여 표시하는 쿼리를 작성합니다. 다음을 볼 수 있어야 합니다:
- 버전 1: London / Retail (만료됨)
- 버전 2: Oxford / Premium (현재)

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I query all SCD Type 2 versions for a specific customer in a Databricks Delta table?"*

**힌트:** `customer_id = 'C-1001'`에서 필터링하고 `ORDER BY valid_from ASC`를 사용합니다.

In [ ]:
%sql
-- Show all versions of customer C-1001, ordered by valid_from
SELECT customer_sk, customer_id, full_name, city, segment, valid_from, valid_to, is_current
FROM banking_lab.silver.dim_customer
WHERE customer_id = 'C-1001'
ORDER BY valid_from ASC;

### 작업 2.6 — 특정 시점 고객 조회

Northbank의 규제 보고팀은 위의 변경이 적용되기 전인 **2020-06-15에 존재했던 고객 프로필**을 알아야 합니다.

`valid_from`과 `valid_to` 열을 사용하여 2020-06-15에 활성화된 버전을 선택하는 고객별 한 행을 반환하는 쿼리를 작성합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I write a point-in-time query on an SCD Type 2 table using valid_from and valid_to date range filtering?"*

**힌트:** 버전은 `valid_from <= D AND valid_to > D`일 때 날짜 `D`에서 활성화되었습니다. 예상 결과: 초기 상태(Oxford 없음, Sophia의 London 없음)의 모든 8명의 원본 고객.

In [ ]:
%sql
-- Return the customer profile as it existed on 2020-06-15
SELECT customer_id, full_name, city, segment, account_type
FROM banking_lab.silver.dim_customer
WHERE valid_from <= TIMESTAMP '2020-06-15 00:00:00'
  AND valid_to   >  TIMESTAMP '2020-06-15 00:00:00'
ORDER BY customer_id;

## 연습 3: 변경 데이터 피드 및 FCA 감사 추적

Basel III 및 FCA 규정에는 Northbank가 거래 데이터에 대한 모든 수정의 **쿼리 가능한 감사 추적**을 유지해야 합니다. 이 연습에서는 결제 거래를 로드하고, 데이터 정정(일반적인 실제 이벤트)을 시뮬레이션하며, **변경 데이터 피드**를 쿼리하여 완전한 감사 로그를 생성합니다.

### 작업 3.1 — 초기 거래 로드

아래 셀을 실행하여 `fact_transactions`에 10개의 결제 거래를 삽입합니다. 이들은 처리된 결제의 1일 배치를 나타냅니다.

In [ ]:
%sql
INSERT INTO banking_lab.silver.fact_transactions VALUES
  ('TXN-001', 'ACC-001', 'C-1001', 'Credit',  1500.00, 'GBP', '2025-11-01', 'Salary payment'),
  ('TXN-002', 'ACC-001', 'C-1001', 'Debit',    120.50, 'GBP', '2025-11-03', 'Amazon purchase'),
  ('TXN-003', 'ACC-002', 'C-1002', 'Debit',    250.00, 'GBP', '2025-11-05', 'Grocery store'),
  ('TXN-004', 'ACC-003', 'C-1003', 'Credit',  3200.00, 'GBP', '2025-11-07', 'Salary payment'),
  ('TXN-005', 'ACC-003', 'C-1003', 'Transfer', 500.00, 'GBP', '2025-11-10', 'Rent transfer'),
  ('TXN-006', 'ACC-004', 'C-1004', 'Debit',     89.99, 'GBP', '2025-11-12', 'Utility bill'),
  ('TXN-007', 'ACC-005', 'C-1005', 'Debit',   1500.00, 'GBP', '2025-11-14', 'Business expense'),
  ('TXN-008', 'ACC-006', 'C-1006', 'Credit',  2800.00, 'GBP', '2025-11-15', 'Salary payment'),
  ('TXN-009', 'ACC-007', 'C-1007', 'Debit',    340.00, 'GBP', '2025-11-18', 'Car insurance'),
  ('TXN-010', 'ACC-008', 'C-1008', 'Transfer', 200.00, 'GBP', '2025-11-20', 'Transfer to savings');

### 작업 3.2 — 거래 정정 시뮬레이션

Northbank의 조정 팀이 2개의 금액 불일치를 발견했습니다. 아래 셀을 실행하여 정정사항을 적용합니다.

In [ ]:
%sql
-- Correction 1: TXN-003 amount was mis-keyed (correct: £275.50, not £250.00)
UPDATE banking_lab.silver.fact_transactions
SET amount = 275.50, description = 'Grocery store (corrected)'
WHERE transaction_id = 'TXN-003';

-- Correction 2: TXN-007 amount is under dispute
UPDATE banking_lab.silver.fact_transactions
SET amount = 1450.00, description = 'Business expense (disputed amount)'
WHERE transaction_id = 'TXN-007';

### 작업 3.3 — 변경 데이터 피드를 쿼리하여 감사 추적 구축

`table_changes()`를 사용하여 `fact_transactions`에 대해 기록된 **모든 변경 이벤트**를 검색하는 쿼리를 작성합니다. `_change_type`, `_commit_version`, `_commit_timestamp` 메타데이터 열을 포함합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I query the Change Data Feed on a Delta table in Databricks using table_changes()? Show change type and commit metadata columns."*

**힌트:** `table_changes('banking_lab.silver.fact_transactions', 0)`를 사용합니다 — 두 번째 인수는 시작 버전 번호입니다.

In [ ]:
%sql
SELECT
    transaction_id,
    amount,
    description,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes('banking_lab.silver.fact_transactions', 0)
ORDER BY _commit_timestamp, transaction_id;

### 작업 3.4 — FCA 보고를 위해 정정된 기록 격리

규정 준수 팀은 수정된 거래의 **정정 후 상태**를 표시하는 뷰가 필요합니다. 변경 데이터 피드를 필터링하여 `update_postimage` 기록만 표시합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I filter table_changes() results in Databricks to show only update_postimage records?"*

**힌트:** 각 `UPDATE` 연산은 두 개의 행을 생성합니다: `update_preimage` (이전) 및 `update_postimage` (변경 후). `_change_type = 'update_postimage'`에 대해 필터링합니다. 예상 결과: TXN-003(£275.50) 및 TXN-007(£1450.00).

In [ ]:
%sql
SELECT
    transaction_id,
    amount,
    description,
    _change_type,
    _commit_timestamp
FROM table_changes('banking_lab.silver.fact_transactions', 0)
WHERE _change_type = 'update_postimage'
ORDER BY _commit_timestamp;

## 연습 4: Delta Lake 시간 여행

Delta Lake는 테이블에 대한 모든 변경의 전체 **거래 로그**를 유지합니다. 이를 통해 **이전 버전의 데이터**를 쿼리하고 배포판을 복원하지 않고도 테이블을 더 이전 상태로 **복원**할 수 있습니다. 이는 거래 데이터가 실수로 수정될 때 Northbank에 매우 중요합니다.

### 작업 4.1 — 거래 기록 검사

`DESCRIBE HISTORY`를 사용하여 `fact_transactions`의 완전한 연산 로그를 봅니다. 출력은 작업 타입, 타임스탬프, 사용자를 포함하여 테이블의 모든 버전을 표시합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I view the full operation history of a Delta table in Databricks, including all version numbers?"*

출력의 **버전 번호**를 참고합니다 — 다음 작업에서 이를 사용합니다.

In [ ]:
%sql
DESCRIBE HISTORY banking_lab.silver.fact_transactions;

### 작업 4.2 — 정정 전의 원본 거래 금액 쿼리

`VERSION AS OF`를 사용하여 **초기 INSERT 이후이지만 정정 전**에 나타났던 `TXN-003` 및 `TXN-007`을 검색하세요.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I query a previous version of a Delta table in Databricks using VERSION AS OF in a SELECT statement?"*

**힌트:** 초기 INSERT는 버전 **1**에 해당합니다 (버전 0은 테이블 생성). 예상 결과: TXN-003과 £250.00, TXN-007과 £1500.00.

In [ ]:
%sql
SELECT transaction_id, amount, description
FROM banking_lab.silver.fact_transactions VERSION AS OF 1
WHERE transaction_id IN ('TXN-003', 'TXN-007');

### 작업 4.3 — 정정 전 상태로 테이블 복원

조정 팀이 정정사항이 오류로 적용되었으며 되돌려야 함을 확인했습니다. `RESTORE TABLE`을 사용하여 `fact_transactions`를 **버전 1**(초기 로드 후, 정정 전)로 되돌립니다.

복원 후 TXN-003 및 TXN-007이 원본 금액을 표시하는지 확인합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I use RESTORE TABLE in Databricks to roll back a Delta table to a specific version number?"*

**힌트:** `RESTORE TABLE banking_lab.silver.fact_transactions TO VERSION AS OF 1;`를 사용합니다

In [ ]:
%sql
RESTORE TABLE banking_lab.silver.fact_transactions TO VERSION AS OF 1;

In [ ]:
%sql
-- Verify: original amounts should be restored
SELECT transaction_id, amount, description
FROM banking_lab.silver.fact_transactions
WHERE transaction_id IN ('TXN-003', 'TXN-007');
-- Expected: TXN-003 = 250.00 'Grocery store', TXN-007 = 1500.00 'Business expense'